In [7]:
from pathlib import Path
import json, numpy as np, pandas as pd
from numpy.linalg import norm

RUN_DIR  = Path("experiments/RN50_20250623_214602")   
IMG_EMB  = np.load(RUN_DIR / "img_embs.npy").astype("float32")
TXT_EMB  = np.load(RUN_DIR / "txt_embs.npy").astype("float32")
IDS      = json.loads((RUN_DIR / "ids.json").read_text())
PARQUET  = Path(r"C:/Users/steph/OneDrive/Desktop/data/metadata.parquet")
META     = pd.read_parquet(PARQUET).set_index("id").loc[IDS]

img_norm = IMG_EMB / norm(IMG_EMB, axis=1, keepdims=True)
txt_norm = TXT_EMB / norm(TXT_EMB, axis=1, keepdims=True)
print("Shapes", img_norm.shape, txt_norm.shape)


Shapes (2000, 1024) (2000, 1024)


In [8]:
def image_to_text_recall(k=1, chunk=2000):
    n = img_norm.shape[0]
    hits = np.zeros(n, dtype=bool)
    for s in range(0, n, chunk):
        e = min(s+chunk, n)
        sim = img_norm[s:e] @ txt_norm.T         # (chunk, N)
        topk = np.argpartition(-sim, k-1, axis=1)[:, :k]
        rows = np.arange(s, e)[:, None]
        hits[s:e] = np.any(topk == rows, axis=1)
    return hits.mean()*100

metrics_it = {f"R@{k}": round(image_to_text_recall(k), 2) for k in (1,5,10)}
metrics_it


{'R@1': 53.25, 'R@5': 78.4, 'R@10': 88.05}

In [9]:
import json
out_file = RUN_DIR / "image_text_metrics.json"
json.dump(metrics_it, open(out_file, "w"), indent=2)
print("Saved", out_file, metrics_it)

Saved experiments\RN50_20250623_214602\image_text_metrics.json {'R@1': 53.25, 'R@5': 78.4, 'R@10': 88.05}


In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from numpy.linalg import norm

RUN_DIR = Path("experiments/ViT-B-32_20250624_170054")

# Load the saved embeddings
IMG_EMB = np.load(RUN_DIR / "img_embs.npy").astype("float32")
TXT_EMB = np.load(RUN_DIR / "txt_embs.npy").astype("float32")

# Load or rebuild the list of IDs
cfg      = json.loads((RUN_DIR / "config.json").read_text())
PARQ     = Path(r"C:/Users/steph/OneDrive/Desktop/data/metadata.parquet")
meta_all = pd.read_parquet(PARQ)
meta_all = meta_all[meta_all["split"]=="train"].reset_index(drop=True)

ids_path = RUN_DIR / "ids.json"
if ids_path.exists():
    IDS = json.loads(ids_path.read_text())
else:
    m = cfg.get("max_samples")
    sub = meta_all.sample(m, random_state=0) if m and m < len(meta_all) else meta_all
    IDS = sub["id"].tolist()    # these are the string IDs
    json.dump(IDS, open(ids_path, "w"))
    print("Rebuilt ids.json")

# Align metadata to the embeddings
if isinstance(IDS[0], int):
    META = meta_all.iloc[IDS].reset_index(drop=True)
else:
    META = meta_all.set_index("id").loc[IDS].reset_index(drop=True)

# Normalize embeddings (L2)
img_n = IMG_EMB / norm(IMG_EMB, axis=1, keepdims=True)
txt_n = TXT_EMB / norm(TXT_EMB, axis=1, keepdims=True)

# Paired image→text Recall@K
def recall_at_k(sim, k):
    topk = np.argpartition(-sim, k-1, axis=1)[:, :k]
    return round(100 * np.mean(np.any(topk == np.arange(sim.shape[0])[:, None], axis=1)), 2)

sim_i2t       = img_n @ txt_n.T
metrics_i2t   = {f"R@{k}": recall_at_k(sim_i2t, k) for k in (1,5,10)}
json.dump(metrics_i2t, open(RUN_DIR/"image_text_metrics.json","w"), indent=2)
print("Saved image_text_metrics.json →", metrics_i2t)

# Cross‐dataset caption Recall@1 (COCO↔Flickr)
domains     = META["domain"].to_numpy()
pos_coco    = np.where(domains=="coco")[0]
pos_flickr  = np.where(domains=="flickr")[0]

def cross_recall(src, tgt):
    if len(src)==0 or len(tgt)==0:
        return 0.0
    sim = src @ tgt.T
    top1 = np.argmax(sim, axis=1)
    return round(100 * np.mean(top1 == np.arange(len(src))), 4)

coco_vec    = txt_n[pos_coco]
flickr_vec  = txt_n[pos_flickr]
metrics_dd  = {
    "coco_flickr_R@1": cross_recall(coco_vec,   flickr_vec),
    "flickr_coco_R@1": cross_recall(flickr_vec, coco_vec),
}
json.dump(metrics_dd, open(RUN_DIR/"dataset2dataset_metrics.json","w"), indent=2)
print("Saved dataset2dataset_metrics.json →", metrics_dd)


Saved image_text_metrics.json → {'R@1': 45.27, 'R@5': 68.45, 'R@10': 76.88}
Saved dataset2dataset_metrics.json → {'coco_flickr_R@1': 0.0, 'flickr_coco_R@1': 0.0}
